[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mongodb-partners/partner-cookbooks/blob/main/advanced-rag/03_graphlookup_and_auto_embedding.ipynb)

# Advanced RAG on MongoDB Atlas: 3. GraphLookup & Auto-Embedding

The final Advanced RAG notebook covers two more MongoDB capabilities that extend
what retrieval can do:

## What you'll learn
1. **`$graphLookup`**: traverse relationships in your data (a movie "knowledge graph" built from
   shared cast), and where graph traversal fits alongside vector and hybrid search.
2. **Auto-Embedding**: let Atlas generate embeddings automatically at write and query time, so
   you never manage an embedding pipeline yourself.

## Prerequisites
Same cluster and credentials as notebooks 1 and 2. This notebook is self-contained otherwise.


## Setup

In [1]:
%%capture
%pip install -q \
    langchain-mongodb==0.11.0 pymongo==4.17.0 python-dotenv==1.2.2 requests==2.34.2

In [2]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

from utils import get_secret, connect

MONGODB_URI  = get_secret("MONGODB_URI",  "MongoDB Atlas connection string: ")
ATLAS_AI_KEY = get_secret("MDB_ATLAS_API_KEY", "Atlas API key: ")
DB_NAME      = os.getenv("MONGODB_DB", "sample_mflix")

client = connect(MONGODB_URI)
db = client[DB_NAME]
movies = db["embedded_movies"]
print("Connected.")

Connected.


## 1. Graph traversal with `$graphLookup`

Vector and full-text search find documents by *content*. Sometimes what you want is *relationships*
like "which movies are connected to this one?" MongoDB's `$graphLookup` stage walks those connections
inside a single aggregation.

Our movies share **cast members**, so we can treat actors as edges: two movies are connected if
they share an actor. Starting from one film, `$graphLookup` follows those edges to find its
neighbourhood in the movie graph.


In [3]:
SEED = "The Terminator"

pipeline = [
    {"$match": {"title": SEED}},
    {"$graphLookup": {
        "from": "embedded_movies",
        "startWith": "$cast",          # begin from this movie's cast
        "connectFromField": "cast",     # follow cast ...
        "connectToField": "cast",       # ... to other movies' cast
        "as": "connected",
        "maxDepth": 0,                  # depth 0 = directly shares an actor
        "depthField": "depth",
    }},
    {"$project": {"_id": 0, "seed": "$title", "seed_cast": "$cast",
                  "connected": {"$slice": ["$connected", 12]}}},
    {"$unwind": "$connected"},
    {"$project": {"seed": 1,
                  "connected_title": "$connected.title",
                  "shared_actors": {"$setIntersection": ["$seed_cast", "$connected.cast"]}}},
    {"$match": {"$expr": {"$ne": ["$seed", "$connected_title"]}}},
    {"$limit": 8},
]

print(f"Movies connected to '{SEED}' through a shared actor:\n")
for r in movies.aggregate(pipeline):
    print(f"  {r['connected_title']:<32} via {', '.join(r['shared_actors'])}")

Movies connected to 'The Terminator' through a shared actor:

  Yellow Rock                      via Michael Biehn
  Batman & Robin                   via Arnold Schwarzenegger
  The Last Stand                   via Arnold Schwarzenegger
  Conan the Destroyer              via Arnold Schwarzenegger
  End of Days                      via Arnold Schwarzenegger
  Escape Plan                      via Arnold Schwarzenegger
  The Expendables 3                via Arnold Schwarzenegger
  The Seventh Sign                 via Michael Biehn


`maxDepth` controls how many hops to follow. Depth 0 finds movies that *directly* share an actor;
increasing it walks further out (actor → movie → co-star → their movies), which is how you'd build
"six degrees" style recommendations. Graph traversal complements semantic search: use vector search
to find a relevant movie, then `$graphLookup` to explore what's related to it.


In [4]:
# How the neighbourhood grows with depth (counts only, to stay fast on M0).
for depth in (0, 1):
    pipe = [
        {"$match": {"title": SEED}},
        {"$graphLookup": {
            "from": "embedded_movies", "startWith": "$cast",
            "connectFromField": "cast", "connectToField": "cast",
            "as": "connected", "maxDepth": depth,
        }},
        {"$project": {"_id": 0, "n": {"$size": "$connected"}}},
    ]
    n = list(movies.aggregate(pipe))[0]["n"]
    print(f"maxDepth={depth}: {n} connected movies")

maxDepth=0: 36 connected movies


maxDepth=1: 259 connected movies


## 2. Auto-Embedding

In notebook 1 we embedded queries ourselves and searched against pre-computed vectors. Atlas can
also **manage embeddings for you**: declare an embedding model on the index, then insert and query
with *plain text*, and Atlas generates the vectors at write and query time. Less pipeline code, no
embedding field to maintain.

We'll build a tiny standalone collection to show it clearly.

> **Free-tier note:** an M0 cluster allows up to 3 search indexes. Notebooks 1 and 2 create two
> (vector and full-text) on `embedded_movies`, so this demo's index is the third. The next cell
> drops any leftover demo index first and waits for the slot to free, so you can re-run the notebook
> without hitting the "maximum number of indexes" limit.


In [5]:
import time

auto = db["auto_embed_demo"]

# Drop any leftover demo collection (and its index) from a previous run, then wait for
# the M0 index slot to actually free before creating a new one.
auto.drop()
time.sleep(45)
db.create_collection("auto_embed_demo")

from langchain_mongodb.index import create_vector_search_index

# For auto-embedding, pass auto_embedding_model and leave dimensions/similarity unset
# (dimensions=-1, similarity=None). Atlas supports the voyage-4 model family here.
create_vector_search_index(
    collection=auto,
    index_name="auto_vec",
    path="plot",
    dimensions=-1,
    similarity=None,
    auto_embedding_model="voyage-4",
    wait_until_complete=300,
)
print("Auto-embedding index is ready.")

Auto-embedding index is ready.


In [6]:
# Insert plain text -- notice there is NO embedding field.
auto.insert_many([
    {"title": "Space Heist", "plot": "Thieves rob a bank orbiting Jupiter."},
    {"title": "Ocean Deep",  "plot": "Divers discover a sea monster in the Mariana Trench."},
    {"title": "Love in Paris", "plot": "Two strangers meet at a cafe and fall in love."},
])
print("Inserted 3 documents as raw text. Atlas is embedding them now.")

Inserted 3 documents as raw text. Atlas is embedding them now.


In [7]:
# Atlas generates embeddings asynchronously after the insert, so we poll until all three
# documents are searchable rather than guessing a fixed wait.
def auto_search(query, expected):
    pipeline = [
        {"$vectorSearch": {
            "index": "auto_vec", "path": "plot",
            "query": query,           # plain text, not "queryVector"
            "numCandidates": 10, "limit": expected,
        }},
        {"$project": {"_id": 0, "title": 1, "score": {"$meta": "vectorSearchScore"}}},
    ]
    for attempt in range(24):        # up to ~3 minutes
        hits = list(auto.aggregate(pipeline))
        if len(hits) >= expected:
            return hits
        time.sleep(8)
    return hits

results = auto_search("a robbery in outer space", expected=3)
for r in results:
    print(f"  {r['title']:<16} score={r['score']:.4f}")

  Space Heist      score=0.5057
  Ocean Deep       score=0.5017
  Love in Paris    score=0.5015


"Space Heist" ranks first for *"a robbery in outer space"*, and we never computed a single
embedding ourselves. Atlas embedded both the documents and the query.

Notice the scores sit close together, near 0.50. With only a few short plots the model has little
to separate them on, so the absolute scores bunch up. The **ranking** is what matters here, and the
right movie comes first. Score spread widens with more documents and longer text; try re-running
the query with different phrasing to see the order change.

**Manual vs. auto-embedding, when to use which:**

| | Manual (notebooks 1 and 2) | Auto-embedding |
|---|---|---|
| Control over model / dimensions | Full | Limited to supported models |
| Pipeline code | You manage embedding | Atlas manages it |
| Existing vectors | Use as-is | Atlas re-embeds from text |
| Best for | Custom models, pre-computed vectors | Getting started fast, less code |


## Recap

Across three notebooks you've built the full advanced-RAG toolkit on a single MongoDB Atlas cluster:
**vector search**, **full-text search**, **hybrid `$rankFusion`**, **reranking**, **`$graphLookup`
traversal**, and **auto-embedding**, with no separate vector database, search engine, or graph store
required.

The Claude Code and MCP notebook shows how to connect these same MongoDB capabilities to AI agents
through the **MongoDB MCP Server**.


In [8]:
# Optional cleanup of the auto-embedding demo collection.
db["auto_embed_demo"].drop()
print("Cleaned up auto_embed_demo.")

Cleaned up auto_embed_demo.
